# S=7 rerun with a longer budget: Transformer AR vs causal encoder, 10,000 updates (4 runs)

The same two S=7 conditions and models as notebook 21. Only the training budget changes: **10,000 updates × 256**
(2.56M examples) instead of 5000. Warmup is still 250 updates, and the cosine schedule now spans 10,000. In notebook
21 the models did not fit the trained k perfectly (AR 98.3–98.7%, causal encoder 99.6–99.8%).

| condition | train k | held-out tests |
|---|---|---|
| S7_k02345 | 0,2,3,4,5 | k=1 at every slot; k=6 (729 words); k=7 (2187 words) |
| S7_k023456 | 0,2,3,4,5,6 | k=1; k=7 |

**Unchanged from notebook 21.** Initialization, logical training stream (the first 5000 batches are identical) and
test sets.

**Evaluation.** The LAST checkpoint, i.e. the model after update 10,000.

**Results.** A new folder, so the 5000-update results are kept. The report includes a 5k vs 10k comparison table.

**Send back `LongK_S7_10k_Review.ipynb`.**


In [ ]:
# --- 2. Bootstrap: locate or clone the PRIVATE repository (credentials are never printed or stored) ---
import base64, importlib.util, os, pathlib, subprocess, sys

REPO_URL = "https://github.com/zhexuandliu/Decompositionality.git"
REPO_BRANCH = "main"
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

def _find_repo():
    candidates = [pathlib.Path("/content/Decompositionality"), pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    for p in candidates:
        if (p / "src" / "decompositionality").is_dir():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    token = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            token = userdata.get("GITHUB_TOKEN")
        except Exception:
            token = None
    if not token:
        import getpass
        token = getpass.getpass("GitHub token with read access to the private repo (input hidden): ")
    # Pass the credential through environment-scoped git config (not the URL, not the command line,
    # not persisted in .git/config).
    _hdr = "AUTHORIZATION: basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
    _env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_CONFIG_COUNT="1",
                GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0=_hdr)
    REPO_DIR = pathlib.Path("/content/Decompositionality" if IN_COLAB else "Decompositionality").resolve()
    _r = subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
                        env=_env, capture_output=True, text=True)
    del token, _hdr, _env
    if _r.returncode != 0:
        raise RuntimeError("git clone failed (check the GITHUB_TOKEN secret / repo access):\n" + _r.stderr[-2000:])

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
for _mod in ("numpy", "pandas", "matplotlib", "torch"):
    if importlib.util.find_spec(_mod) is None:  # Colab already ships these; never reinstall torch/CUDA here
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _mod], check=True)
REPO_COMMIT = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Repository:", REPO_DIR, "| commit:", REPO_COMMIT or "unknown")


In [ ]:
# --- CONFIGURATION -------------------------------------------------------------------------------------------------
MODE = os.environ.get("DECOMP_MODE", "full")
SEED = 42
RUN_CONDITIONS = ["S7_k02345", "S7_k023456"]
RUN_MODELS = ["transformer_ar", "causal_encoder"]
STEPS = 10000
SUBDIR = "longk_ar_vs_causal_encoder_S7_10k_v1"     # separate from notebook 21's results
N_TEST_GRIDS = 1000
K7_GRIDS = 300
USE_DRIVE = IN_COLAB
RESULTS_ROOT = os.environ.get("DECOMP_RESULTS_ROOT") or (
    "/content/drive/MyDrive/decompositionality_results" if USE_DRIVE else str(REPO_DIR / "results"))
DEVICE = os.environ.get("DECOMP_DEVICE", "auto")
# -----------------------------------------------------------------------------------------------------------------
if USE_DRIVE and IN_COLAB and RESULTS_ROOT.startswith("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
import pandas as pd
from IPython.display import display
pd.set_option("display.max_rows", 500); pd.set_option("display.max_columns", 60); pd.set_option("display.width", 250)
from decompositionality import longk as LK, longk_exp as LX
_c = LX.make_cfg(RUN_CONDITIONS[0], RESULTS_ROOT, MODE, RUN_MODELS, SEED, SUBDIR,
                 **({"optimizer_steps": STEPS} if MODE == "full" else {}))
print("resolved recipe:", _c.train.semantic_dict())
print("results ->", LX.family_root(RESULTS_ROOT, MODE, SUBDIR))


In [ ]:
# --- Train (10,000 updates) + evaluate the LAST checkpoint (resumable) ----------------------------------------------
status = LX.run_queue(RESULTS_ROOT, MODE, SEED, RUN_CONDITIONS, RUN_MODELS, repo_dir=str(REPO_DIR),
                      n_grids=N_TEST_GRIDS, k7_grids=K7_GRIDS, device=DEVICE, subdir=SUBDIR,
                      steps=STEPS if MODE == "full" else None)
display(status)


In [ ]:
# --- Report, incl. 5k (notebook 21) vs 10k comparison; reads saved artifacts only ---------------------------------
rp = LX.report(RESULTS_ROOT, MODE, SEED, conditions=RUN_CONDITIONS, subdir=SUBDIR, compare_subdir=LK.SUBDIR,
               title_note=f"Budget: {STEPS} updates.", review_name="LongK_S7_10k_Review.ipynb")
print("REVIEW FILE TO SEND BACK:", rp)
if IN_COLAB:
    from google.colab import files
    files.download(str(rp))
